# FMCG Capital Structure — `process.ipynb`

**Data-collection workbench.** Imports → initialise config → make the Screener API call
and inspect what comes back. One page fetch per firm yields **two** datasets:
the current market snapshot **and** the multi-year P&L / Balance Sheet / Cash Flow tables.

> Heavy logic lives in `analysis/scripts/` (`config.py`, `scrape_market_data.py`) — this
> notebook just drives and inspects it (Instruction #3). ⚠ The API-call cells are written
> but **not run** — you trigger them yourself to see the raw data (Instruction #4).

## 0 · Imports & setup

In [ ]:
# --- stdlib ---
import sys
import time
from pathlib import Path

# --- third party ---
import requests
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"python   : {sys.version.split()[0]}")
print(f"pandas   : {pd.__version__}")
print(f"requests : {requests.__version__}")

In [ ]:
# put analysis/scripts on the path so we can reuse the background functions, not re-write them
SCRIPTS_DIR = Path.cwd().parents[1] / "analysis" / "scripts"   # notebooks/ -> analysis -> root
if not (SCRIPTS_DIR / "config.py").exists():                    # fallback: notebook opened from root
    SCRIPTS_DIR = Path.cwd() / "analysis" / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

import config                      # constants: companies, URLs, paths, assumptions
import scrape_market_data as smd   # fetch_page / parse_top_ratios / parse_data_table / scrape_company

print(f"scripts dir  : {SCRIPTS_DIR}")
print(f"project root : {config.PROJECT_ROOT}")
config.COMPANIES

## 1 · Initialise — config snapshot

In [ ]:
# the 8-firm universe and where each one lives on Screener (aligned for scanning)
for _name in config.COMPANIES:
    print(f"  {_name:8s} {config.NSE_SYMBOL[_name]:12s} {config.SCREENER_URL[_name]}")

print(f"\nperiod : {config.YEARS}   ({len(config.YEARS)} fiscal years)")
print(f"betas  : {config.BETA}")

In [ ]:
# sanity-check the scaffold exists before we write anything (diagnostic)
for _label, _path in [("raw    ", config.DATA_RAW),
                      ("cleaned", config.DATA_CLEANED),
                      ("output ", config.OUTPUT_DIR)]:
    _mark = "✓" if _path.exists() else "⚠ missing"
    print(f"  {_label}: {_mark}  {_path}")

## 2 · Screener API call  ⚠ you run this

One fetch per firm → **snapshot** (`market_data.csv`) + **multi-year statements**
(`financials.csv`). We probe one company first to eyeball both structures, then loop all
eight. Screener may serve a login wall to bare requests — if so the parse comes back empty
and we fall back to the alternatives in `docs/data_sources.md`.

> Enterprise Value is **not** here — Screener doesn't expose it. EV is computed later as
> `Market Cap + Total Debt - Cash` from the statement data.

In [ ]:
# step 1 — probe ONE company so we can see exactly what Screener returns
test_name  = "HUL"
html_0_hul = smd.fetch_page(config.SCREENER_URL[test_name])
print("fetched chars:", len(html_0_hul) if html_0_hul else None)

In [ ]:
# step 2 — snapshot: parse the headline ratios block; render the raw dict to inspect it
ratios_0_hul = smd.parse_top_ratios(html_0_hul) if html_0_hul else {}
ratios_0_hul

In [ ]:
# step 2b — statements: parse ONE table (P&L) to eyeball the multi-year (year-column) shape
pl_0_hul = smd.parse_data_table(html_0_hul, "profit-loss") if html_0_hul else []
pd.DataFrame(pl_0_hul).head(12)

In [ ]:
# step 3 — loop all eight; each page is fetched ONCE and yields BOTH datasets
market_rows_0, fin_records_0 = [], []
for _name in config.COMPANIES:
    _mrow, _fin = smd.scrape_company(_name, config.SCREENER_URL[_name])
    market_rows_0.append(_mrow)
    fin_records_0.extend(_fin)
    time.sleep(smd.POLITE_DELAY)                # polite gap between hits

market_df_0 = pd.DataFrame(market_rows_0)
fin_df_0    = pd.DataFrame(fin_records_0)
fin_df_0.head()

In [ ]:
# step 4 — sanity check: did we get the years? how many cells per firm / section?
print(f"market snapshot : {market_df_0.shape}")
print(f"statements      : {fin_df_0.shape}")
if not fin_df_0.empty:
    print("periods         :", sorted(fin_df_0["Period"].unique()))
    print("cells per firm  :")
    for _co, _n in fin_df_0.groupby("Company").size().items():
        print(f"  {_co:8s} {_n}")
fin_df_0.groupby(["Company", "Section"]).size().unstack(fill_value=0)

In [ ]:
# step 5 — persist both datasets to data/raw/ (overwrites)
config.DATA_RAW.mkdir(parents=True, exist_ok=True)
market_df_0.to_csv(config.MARKET_DATA_CSV, index=False)
fin_df_0.to_csv(config.FINANCIALS_CSV, index=False)
print(f"✓ wrote {config.MARKET_DATA_CSV}")
print(f"✓ wrote {config.FINANCIALS_CSV}")

### Alternative API — Yahoo Finance (kept as a fallback, not run)

If Screener blocks the scrape, `yfinance` pulls the same data cleanly: uncomment
`yfinance` in `requirements.txt`, `pip install -r requirements.txt`, then run the cell
below. (`.financials` / `.balance_sheet` / `.cashflow` give the multi-year statements.)

In [ ]:
# --- alternative: yfinance (archival; uncomment to use — needs `pip install yfinance`) ---
# import yfinance as yf
# _t = yf.Ticker(config.YAHOO_TICKER["HUL"])
# print(_t.info.get("marketCap"), _t.info.get("enterpriseValue"))
# _t.financials      # multi-year income statement (years as columns)
# _t.balance_sheet   # multi-year balance sheet
# _t.cashflow        # multi-year cash flow

---
**Next move (after you run the call & confirm the data):**
`clean_data.py` (reshape `financials.csv` → 40-row `master_data.csv`) →
`compute_ratios.py` → `leverage_threshold.py` → the formula-driven Excel workbook.

## 3 · Cleaned CSVs → one workbook (sheet per `Company_Section`)

Gather the 24 cleaned frames under `data/cleaned/<Company>/<Section>.csv` into a single
Excel workbook — one sheet each, named `<Company>_<Section>` (e.g. `HUL_P&L`).

In [ ]:
# write each cleaned CSV to its own sheet named "<Company>_<Section>" in one workbook
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(config.WORKBOOK_XLSX, engine="openpyxl") as xl:
    for _company in config.COMPANIES:
        for _csv in sorted((config.DATA_CLEANED / _company).glob("*.csv")):
            _section = _csv.stem                              # 'P&L' / 'BalanceSheet' / 'CashFlow'
            _sheet   = f"{_company}_{_section}"               # 'HUL_P&L'
            _df      = pd.read_csv(_csv, index_col=0)         # Period back as the index
            _df.to_excel(xl, sheet_name=_sheet)
            print(f"  {_sheet:22s} {_df.shape[0]}x{_df.shape[1]}")

print(f"\n✓ wrote {config.WORKBOOK_XLSX}")